# Phase 4: Experimental ThunderSVM Integration (High Performance)
**CSC14120 - Parallel Programming**

## Objective
Attempt to use **ThunderSVM** (GPU-accelerated SVM) to train on a larger dataset than CPU-based sklearn, targeting **Extra Credit**.

## Strategy for Stability
1. **Custom Build:** Compile ThunderSVM specifically for the current GPU architecture.
2. **Memory Management:**
   - Aggressive garbage collection.
   - Dynamic sample sizing (start with 25k, reduce if crash).
   - 32-bit float precision (standard).

## Risk
ThunderSVM allows training on 20k-50k samples in seconds, but is prone to CUDA errors on Colab due to environment mismatches. This notebook implements safeguards.

In [ ]:
# Check GPU and CUDA version
!nvidia-smi
!nvcc --version

In [ ]:
# Install dependencies & build tools
!apt-get install -y cmake
import sys
import os
import gc

In [ ]:
# Upload and extract project
from google.colab import files
import zipfile
import os

if not os.path.exists('src'):
    print("Upload file zip project:")
    uploaded = files.upload()
    if uploaded:
        zip_name = list(uploaded.keys())[0]
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall('project')
        
        for root, dirs, _ in os.walk('project'):
            if 'src' in dirs:
                os.chdir(root)
                break
else:
    print("Project already exists.")

print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Download CIFAR-10
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/

print('CIFAR-10 ready!')

In [ ]:
# Build ThunderSVM from Source (Optimized for Colab T4/P100)
print("Building ThunderSVM from source...")

# Cleanup previous attempts
!rm -rf thundersvm

# Clone
!git clone --depth 1 https://github.com/Xtra-Computing/thundersvm.git

# Build with explicit CUDA support
# We force standard cmake build which usually finds the right CUDA on Colab
!cd thundersvm && mkdir -p build && cd build && cmake .. -DUSE_CUDA=ON -DCMAKE_BUILD_TYPE=Release && make -j4

# Install Python bindings
!cd thundersvm/python && pip install -e .

# Verify import
try:
    sys.path.insert(0, 'thundersvm/python')
    from thundersvm import SVC as ThunderSVC
    print("ThunderSVM installed successfully!")
except ImportError as e:
    print(f"Failed to import ThunderSVM: {e}")

In [ ]:
# Build Feature Extractor (CUDA C++)
# Requires LIBSVM for compilation compatibility

!git clone --depth 1 https://github.com/cjlin1/libsvm.git libsvm_src 2>/dev/null || echo "LIBSVM present"
!cd libsvm_src && make lib
!cp libsvm_src/svm.h include/ 2>/dev/null
!cp libsvm_src/svm.cpp src/ 2>/dev/null

import os
print("Building feature_extractor...")
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DWITH_SVM -DWITH_LIBSVM \
    -Iinclude -Ilibsvm_src -lcublas -lcudnn \
    -o feature_extractor \
    src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu \
    src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp libsvm_src/svm.cpp

if os.path.exists('feature_extractor'):
    print("Feature Extractor Build SUCCESS!")
else:
    print("Feature Extractor Build FAILED!")

In [ ]:
# Upload Phase 3 Weights if missing
weights_file = 'phase3_opt.weights'
if not os.path.exists(weights_file):
    print(f"Upload {weights_file}:")
    uploaded = files.upload()
else:
    print(f"Weights found: {weights_file}")

In [ ]:
# Extract Features (Extract Only Mode)
import time

print("Extracting features...")
start = time.time()

# This saves 'train_features.bin' and 'test_features.bin'
!./feature_extractor --data data --weights phase3_opt.weights --extract-only

print(f"Extraction time: {time.time() - start:.2f}s")

In [ ]:
# Load Data & Prepare
import numpy as np
import struct
from sklearn.preprocessing import normalize

feature_dim = 8192

def load_labels(data_dir):
    train_labels = []
    test_labels = []
    for i in range(1, 6):
        with open(f"{data_dir}/data_batch_{i}.bin", 'rb') as f:
            for _ in range(10000):
                train_labels.append(struct.unpack('B', f.read(1))[0])
                f.read(3072)
    with open(f"{data_dir}/test_batch.bin", 'rb') as f:
        for _ in range(10000):
            test_labels.append(struct.unpack('B', f.read(1))[0])
            f.read(3072)
    return np.array(train_labels), np.array(test_labels)

print("Loading features & labels...")
X_train_full = np.fromfile('train_features.bin', dtype=np.float32).reshape(-1, feature_dim)
X_test = np.fromfile('test_features.bin', dtype=np.float32).reshape(-1, feature_dim)
y_train_full, y_test = load_labels('data')

# Normalize (Critical for SVM)
print("Normalizing...")
X_train_full = normalize(X_train_full, norm='l2')
X_test = normalize(X_test, norm='l2')

print(f"Full Train set: {X_train_full.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Dynamic Training Loop
# Try largest possible sample size, reduce if it crashes/fails

def train_evaluate(n_samples):
    print(f"\n--- Attempting training with {n_samples} samples ---")
    
    # Stratified subset
    samples_per_class = n_samples // 10
    indices = []
    for c in range(10):
        class_idx = np.where(y_train_full == c)[0]
        indices.extend(class_idx[:samples_per_class])
    
    indices = np.array(indices)
    X_subset = X_train_full[indices]
    y_subset = y_train_full[indices]
    
    print(f"Subset shape: {X_subset.shape}")
    
    # Train
    try:
        gc.collect() # Free memory
        
        # ThunderSVM parameters
        # gpu_id=0, kernel='rbf', C=10, gamma='auto'
        svm = ThunderSVC(kernel='rbf', C=10.0, gamma='auto', gpu_id=0, verbose=True)
        
        start = time.time()
        svm.fit(X_subset, y_subset)
        train_time = time.time() - start
        print(f"Training SUCCESS in {train_time:.2f}s")
        
        # Evaluate
        acc = svm.score(X_test, y_test)
        print(f"Test Accuracy: {acc*100:.2f}%")
        return True, acc, train_time
        
    except Exception as e:
        print(f"Training FAILED with {n_samples} samples: {e}")
        return False, 0.0, 0.0

# Try decreasing sizes: 40k -> 30k -> 20k -> 10k
sample_sizes = [40000, 30000, 20000, 10000]

best_acc = 0.0

for size in sample_sizes:
    success, acc, t = train_evaluate(size)
    if success:
        best_acc = acc
        break # Stop if we found a working size (prefer larger)
    
if best_acc == 0.0:
    print("\nAll ThunderSVM attempts failed. Please use the CPU notebook.")

In [ ]:
# Visualization (if successful)
if best_acc > 0:
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import confusion_matrix
    
    # Re-predict for confusion matrix
    y_pred = svm.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'ThunderSVM Confusion Matrix (Acc: {best_acc*100:.2f}%)')
    plt.show()